# Build a Transformer: Name Generation

In this notebook we build a **character-level transformer** for generating names, step by step.
We start from the simplest possible model (a bigram count table) and work our way up to a full
transformer with multi-head self-attention, residual connections, and layer normalization.

This notebook accompanies the lesson on the [mlearn website](https://mlearn.dev).

**Models we will build:**
1. Bigram (counting)
2. Neural Bigram (single linear layer)
3. MLP Language Model
4. Single-Head / Multi-Head Self-Attention
5. Full Transformer

## 1. Setup

In [ ]:
!pip install torch matplotlib -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random

# reproducibility
torch.manual_seed(42)
random.seed(42)

print(f'PyTorch version: {torch.__version__}')

## 2. Dataset & Character Encoding

We use a hardcoded list of common names. Each name is a sequence of characters;
we add a special `'.'` token to mark the start and end of a name.

In [ ]:
names = [
    "emma", "olivia", "ava", "sophia", "isabella", "mia", "charlotte", "amelia",
    "harper", "evelyn", "abigail", "emily", "elizabeth", "sofia", "ella", "madison",
    "scarlett", "victoria", "aria", "grace", "chloe", "camila", "penelope", "riley",
    "layla", "lillian", "nora", "zoey", "mila", "aubrey", "hannah", "lily", "addison",
    "eleanor", "natalie", "luna", "savannah", "brooklyn", "leah", "zoe", "stella",
    "hazel", "ellie", "paisley", "audrey", "skylar", "violet", "claire", "bella",
    "aurora", "liam", "noah", "oliver", "james", "elijah", "william", "henry",
    "lucas", "benjamin", "theodore", "jack", "levi", "alexander", "mason", "ethan",
    "daniel", "jacob", "michael", "sebastian", "owen", "aiden", "samuel", "ryan",
    "nathan", "caleb", "christian", "dylan", "landon", "josiah", "andrew", "david",
    "adrian", "miles", "eli", "nolan", "hunter", "isaiah", "thomas", "aaron",
    "roman", "colton", "leo", "connor", "ezekiel", "hudson", "easton", "asher",
    "robert", "xavier", "adam", "jose", "ivan", "clara", "elena", "iris"
]

print(f'Number of names: {len(names)}')
print(f'Examples: {names[:10]}')

In [ ]:
# Build character vocabulary
chars = sorted(list(set(''.join(names))))
chars = ['.'] + chars  # '.' is our start/end token (index 0)
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

print(f'Vocabulary ({vocab_size} chars): {chars}')
print(f'stoi example: a -> {stoi["a"]}, z -> {stoi["z"]}')

In [ ]:
# Train / validation split
random.shuffle(names)
n = int(0.9 * len(names))
train_names = names[:n]
val_names = names[n:]
print(f'Train: {len(train_names)}, Val: {len(val_names)}')

## 3. Bigram Model (Counting)

The simplest language model: just count how often character *b* follows character *a*,
then normalize to get probabilities.

In [ ]:
# Build bigram count matrix
N = torch.zeros((vocab_size, vocab_size), dtype=torch.int32)

for name in train_names:
    chs = ['.'] + list(name) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        N[stoi[ch1], stoi[ch2]] += 1

print('Count matrix shape:', N.shape)
print('Total bigrams:', N.sum().item())

In [ ]:
# Visualize the bigram counts
fig, ax = plt.subplots(figsize=(16, 16))
ax.imshow(N.float(), cmap='Blues')
for i in range(vocab_size):
    for j in range(vocab_size):
        label = itos[i] + itos[j]
        ax.text(j, i, label, ha='center', va='bottom', fontsize=6, color='gray')
        ax.text(j, i, N[i, j].item(), ha='center', va='top', fontsize=6)
ax.set_title('Bigram Counts')
plt.tight_layout()
plt.show()

In [ ]:
# Probability matrix (add smoothing to avoid zero probs)
P = (N + 1).float()
P = P / P.sum(dim=1, keepdim=True)

# Compute negative log-likelihood on val set
log_likelihood = 0.0
count = 0
for name in val_names:
    chs = ['.'] + list(name) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        prob = P[stoi[ch1], stoi[ch2]]
        log_likelihood += torch.log(prob)
        count += 1

bigram_count_loss = -log_likelihood / count
print(f'Bigram (count) val NLL: {bigram_count_loss.item():.4f}')

In [ ]:
# Generate names from the bigram count model
g = torch.Generator().manual_seed(42)

print('Bigram (count) generated names:')
for _ in range(10):
    out = []
    ix = 0  # start token
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print('  ' + ''.join(out))

## 4. Neural Bigram Model

Replace the explicit count table with a single linear layer.
The input is a one-hot vector (or equivalently, an embedding lookup),
and the output is logits over the next character.

In [ ]:
# Build training data for bigram: (prev_char) -> (next_char)
def build_bigram_dataset(name_list):
    xs, ys = [], []
    for name in name_list:
        chs = ['.'] + list(name) + ['.']
        for ch1, ch2 in zip(chs, chs[1:]):
            xs.append(stoi[ch1])
            ys.append(stoi[ch2])
    return torch.tensor(xs), torch.tensor(ys)

xtr_bi, ytr_bi = build_bigram_dataset(train_names)
xval_bi, yval_bi = build_bigram_dataset(val_names)
print(f'Train bigrams: {len(xtr_bi)}, Val bigrams: {len(xval_bi)}')

In [ ]:
# Neural bigram: a single linear layer (logits = W[input_char])
W = torch.randn((vocab_size, vocab_size), requires_grad=True, generator=torch.Generator().manual_seed(42))

# Training loop
for step in range(200):
    # forward
    xenc = F.one_hot(xtr_bi, num_classes=vocab_size).float()
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(dim=1, keepdim=True)
    loss = -probs[torch.arange(len(ytr_bi)), ytr_bi].log().mean()
    # add regularization
    loss += 0.01 * (W ** 2).mean()

    # backward
    W.grad = None
    loss.backward()

    # update
    W.data -= 50 * W.grad

    if step % 50 == 0:
        print(f'Step {step:4d} | loss: {loss.item():.4f}')

# Validation loss
with torch.no_grad():
    xenc = F.one_hot(xval_bi, num_classes=vocab_size).float()
    logits = xenc @ W
    neural_bigram_loss = F.cross_entropy(logits, yval_bi)
    print(f'\nNeural bigram val loss: {neural_bigram_loss.item():.4f}')

In [ ]:
# Generate names from the neural bigram model
g = torch.Generator().manual_seed(42)
print('Neural bigram generated names:')
for _ in range(10):
    out = []
    ix = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=vocab_size).float()
        logits = xenc @ W
        p = F.softmax(logits, dim=1)
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print('  ' + ''.join(out))

## 5. MLP Language Model

Now we look at more context than just one character. We use a context window
of `block_size` characters, embed them, concatenate, and pass through a
hidden layer with tanh activation.

In [ ]:
block_size = 3  # context length

def build_mlp_dataset(name_list):
    X, Y = [], []
    for name in name_list:
        context = [0] * block_size  # start with all '.' tokens
        for ch in name + '.':
            ix = stoi[ch]
            X.append(context[:])
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

Xtr, Ytr = build_mlp_dataset(train_names)
Xval, Yval = build_mlp_dataset(val_names)
print(f'Train: X={Xtr.shape}, Y={Ytr.shape}')
print(f'Val:   X={Xval.shape}, Y={Yval.shape}')

# Show a few examples
for i in range(5):
    ctx = ''.join(itos[c.item()] for c in Xtr[i])
    print(f'  {ctx} --> {itos[Ytr[i].item()]}')

In [ ]:
# MLP model
n_embd = 10
n_hidden = 200

g = torch.Generator().manual_seed(42)
C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((block_size * n_embd, n_hidden), generator=g) * (5/3) / (block_size * n_embd)**0.5
b1 = torch.randn(n_hidden, generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01
b2 = torch.randn(vocab_size, generator=g) * 0

parameters_mlp = [C, W1, b1, W2, b2]
for p in parameters_mlp:
    p.requires_grad = True

print(f'MLP parameters: {sum(p.nelement() for p in parameters_mlp)}')

In [ ]:
# Training loop
mlp_losses = []
for step in range(10000):
    # mini-batch
    ix = torch.randint(0, Xtr.shape[0], (64,))
    Xb, Yb = Xtr[ix], Ytr[ix]

    # forward
    emb = C[Xb]  # (B, block_size, n_embd)
    h = torch.tanh(emb.view(-1, block_size * n_embd) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    # backward
    for p in parameters_mlp:
        p.grad = None
    loss.backward()

    # update
    lr = 0.1 if step < 5000 else 0.01
    for p in parameters_mlp:
        p.data -= lr * p.grad

    if step % 2000 == 0:
        print(f'Step {step:5d} | loss: {loss.item():.4f}')
    mlp_losses.append(loss.item())

# Validation loss
with torch.no_grad():
    emb = C[Xval]
    h = torch.tanh(emb.view(-1, block_size * n_embd) @ W1 + b1)
    logits = h @ W2 + b2
    mlp_val_loss = F.cross_entropy(logits, Yval)
    print(f'\nMLP val loss: {mlp_val_loss.item():.4f}')

In [ ]:
# Generate names from the MLP
g = torch.Generator().manual_seed(42)
print('MLP generated names:')
for _ in range(10):
    out = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        p = F.softmax(logits, dim=1)
        ix = torch.multinomial(p, num_samples=1, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
        context = context[1:] + [ix]
    print('  ' + ''.join(out))

## 6. Self-Attention

Before building the full transformer, let us understand self-attention.
Each position produces a **query** and a **key**; the dot product of queries and keys
determines how much each position attends to every other. A **causal mask** ensures
we only look at the past.

In [ ]:
# Demonstrate single-head self-attention on a small example
torch.manual_seed(42)

B, T, C_dim = 4, block_size, n_embd  # batch, time, channels
head_size = 16

# Random input (pretend these are embeddings)
x = torch.randn(B, T, C_dim)

key = nn.Linear(C_dim, head_size, bias=False)
query = nn.Linear(C_dim, head_size, bias=False)
value = nn.Linear(C_dim, head_size, bias=False)

k = key(x)    # (B, T, head_size)
q = query(x)  # (B, T, head_size)
v = value(x)  # (B, T, head_size)

# Attention scores
wei = q @ k.transpose(-2, -1) * head_size**-0.5  # (B, T, T)

# Causal mask: can only attend to current and previous positions
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

out = wei @ v  # (B, T, head_size)

print('Attention weights (first example):')
print(wei[0].detach())
print(f'\nOutput shape: {out.shape}')

In [ ]:
# Multi-head attention: run several heads in parallel and concatenate

class Head(nn.Module):
    """One head of self-attention."""
    def __init__(self, n_embd, head_size, block_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_heads, head_size, block_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size) for _ in range(n_heads)])
        self.proj = nn.Linear(n_heads * head_size, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)

# Quick test
mha = MultiHeadAttention(n_embd=10, n_heads=2, head_size=8, block_size=3)
test_in = torch.randn(2, 3, 10)
test_out = mha(test_in)
print(f'Multi-head attention: input {test_in.shape} -> output {test_out.shape}')

## 7. Full Transformer Model

Now we assemble everything: token embeddings, positional embeddings,
transformer blocks (multi-head attention + feed-forward, each with
residual connections and layer norm), and a final linear head.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_heads, block_size):
        super().__init__()
        head_size = n_embd // n_heads
        self.sa = MultiHeadAttention(n_embd, n_heads, head_size, block_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))   # residual + attention
        x = x + self.ffwd(self.ln2(x)) # residual + feed-forward
        return x


class CharTransformer(nn.Module):
    def __init__(self, vocab_size, n_embd, n_heads, n_layers, block_size):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[TransformerBlock(n_embd, n_heads, block_size) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_emb(idx)  # (B, T, n_embd)
        pos_emb = self.pos_emb(torch.arange(T, device=idx.device))  # (T, n_embd)
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, V = logits.shape
            logits_flat = logits.view(B * T, V)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]  # crop to block_size
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]  # last time step
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx


# Hyperparameters
tf_block_size = 8
tf_n_embd = 64
tf_n_heads = 4
tf_n_layers = 2

model = CharTransformer(vocab_size, tf_n_embd, tf_n_heads, tf_n_layers, tf_block_size)
print(f'Transformer parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Build dataset for the transformer (input sequence -> next char at each position)
def build_transformer_dataset(name_list, block_size):
    X, Y = [], []
    for name in name_list:
        seq = [0] + [stoi[ch] for ch in name] + [0]  # . name .
        for i in range(len(seq) - 1):
            # Grab up to block_size input chars and the corresponding targets
            start = max(0, i - block_size + 1)
            inp = seq[start:i+1]
            tgt = seq[start+1:i+2]
            # Pad on the left if shorter than block_size
            pad_len = block_size - len(inp)
            inp = [0] * pad_len + inp
            tgt = [0] * pad_len + tgt
            X.append(inp)
            Y.append(tgt)
    return torch.tensor(X), torch.tensor(Y)

Xtr_tf, Ytr_tf = build_transformer_dataset(train_names, tf_block_size)
Xval_tf, Yval_tf = build_transformer_dataset(val_names, tf_block_size)
print(f'Train: X={Xtr_tf.shape}, Y={Ytr_tf.shape}')
print(f'Val:   X={Xval_tf.shape}, Y={Yval_tf.shape}')

In [ ]:
# Training loop for the transformer
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
tf_train_losses = []

for step in range(5000):
    # mini-batch
    ix = torch.randint(0, Xtr_tf.shape[0], (64,))
    Xb, Yb = Xtr_tf[ix], Ytr_tf[ix]

    logits, loss = model(Xb, Yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 1000 == 0:
        print(f'Step {step:5d} | loss: {loss.item():.4f}')
    tf_train_losses.append(loss.item())

# Reduce learning rate and train more
for g_opt in optimizer.param_groups:
    g_opt['lr'] = 1e-3

for step in range(5000):
    ix = torch.randint(0, Xtr_tf.shape[0], (64,))
    Xb, Yb = Xtr_tf[ix], Ytr_tf[ix]
    logits, loss = model(Xb, Yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 1000 == 0:
        print(f'Step {step+5000:5d} | loss: {loss.item():.4f}')
    tf_train_losses.append(loss.item())

# Validation loss
with torch.no_grad():
    _, tf_val_loss = model(Xval_tf, Yval_tf)
    print(f'\nTransformer val loss: {tf_val_loss.item():.4f}')

In [ ]:
# Generate names from the transformer
torch.manual_seed(42)
print('Transformer generated names:')
for _ in range(15):
    idx = torch.zeros((1, 1), dtype=torch.long)  # start with '.'
    generated = model.generate(idx, max_new_tokens=15)
    tokens = generated[0].tolist()
    name = ''
    for t in tokens[1:]:  # skip the initial '.'
        if t == 0:
            break
        name += itos[t]
    print('  ' + name)

## 8. Comparison

Let us compare all models by their validation loss and look at sample outputs.

In [ ]:
# Summary of validation losses
print('=== Validation Losses (NLL) ===')
print(f'  Bigram (count):   {bigram_count_loss.item():.4f}')
print(f'  Neural Bigram:    {neural_bigram_loss.item():.4f}')
print(f'  MLP:              {mlp_val_loss.item():.4f}')
print(f'  Transformer:      {tf_val_loss.item():.4f}')

# Bar chart
model_names = ['Bigram\n(count)', 'Neural\nBigram', 'MLP', 'Transformer']
losses = [bigram_count_loss.item(), neural_bigram_loss.item(),
          mlp_val_loss.item(), tf_val_loss.item()]
colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(model_names, losses, color=colors, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Validation Loss (NLL)')
ax.set_title('Model Comparison: Validation Loss')
for bar, loss in zip(bars, losses):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{loss:.3f}', ha='center', va='bottom', fontweight='bold')
ax.set_ylim(0, max(losses) * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
# Training loss curves (MLP and Transformer)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# MLP loss curve (smoothed)
window = 100
mlp_smooth = [sum(mlp_losses[i:i+window])/window
              for i in range(0, len(mlp_losses)-window, window)]
ax1.plot(range(0, len(mlp_losses)-window, window), mlp_smooth, color='#e15759')
ax1.set_title('MLP Training Loss')
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')

# Transformer loss curve (smoothed)
tf_smooth = [sum(tf_train_losses[i:i+window])/window
             for i in range(0, len(tf_train_losses)-window, window)]
ax2.plot(range(0, len(tf_train_losses)-window, window), tf_smooth, color='#76b7b2')
ax2.set_title('Transformer Training Loss')
ax2.set_xlabel('Step')
ax2.set_ylabel('Loss')

plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side sample names from each model
print('=== Sample Generated Names ===')
print()

# Bigram (count)
g = torch.Generator().manual_seed(123)
print('Bigram (count):')
for _ in range(5):
    out = []
    ix = 0
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print(f'  {".".join([]) + "".join(out)}')

# Neural Bigram
g = torch.Generator().manual_seed(123)
print('\nNeural Bigram:')
for _ in range(5):
    out = []
    ix = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=vocab_size).float()
        logits = xenc @ W
        p = F.softmax(logits, dim=1)
        ix = torch.multinomial(p, num_samples=1, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print(f'  {"".join(out)}')

# MLP
g = torch.Generator().manual_seed(123)
print('\nMLP:')
for _ in range(5):
    out = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        p = F.softmax(logits, dim=1)
        ix = torch.multinomial(p, num_samples=1, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
        context = context[1:] + [ix]
    print(f'  {"".join(out)}')

# Transformer
torch.manual_seed(123)
print('\nTransformer:')
for _ in range(5):
    idx = torch.zeros((1, 1), dtype=torch.long)
    generated = model.generate(idx, max_new_tokens=15)
    tokens = generated[0].tolist()
    name = ''
    for t in tokens[1:]:
        if t == 0:
            break
        name += itos[t]
    print(f'  {name}')

print('\nDone! Visit mlearn.dev for the full lesson.')